# Module 3: Window Functions Deep Dive

**Objective**: Master window functions for banking analytics - running balances, rankings, moving averages.

## Key Concepts
- `Window.partitionBy()` - Group rows for the window
- `Window.orderBy()` - Order within each partition
- `rowsBetween()` / `rangeBetween()` - Define window frame

In [1]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module03-WindowFunctions").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

✅ Databricks Connect | Spark 4.1.0
Mode: databricks


In [4]:
from pyspark.sql import functions as F
# ── Load data (S3 Parquet or local CSV) ──
transactions_df = spark.read.format('delta').load(f"{S3_RAW}/transactions_delta") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
accounts_df = spark.read.parquet(f"{S3_RAW}/accounts") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)

txn = transactions_df.withColumn("txn_date", F.to_date(F.col("txn_datetime")))
txn.show(5)

+----------+----------+-------------------+-----------+-------------+--------+----------------+-----------------+---------+------------+--------------------+----------+
|    txn_id|account_id|       txn_datetime|   txn_type|       amount|currency|         channel|merchant_category|   status|   reference|         description|  txn_date|
+----------+----------+-------------------+-----------+-------------+--------+----------------+-----------------+---------+------------+--------------------+----------+
|TXN0002838|ACCT000740|2025-11-04 21:19:19|        Fee|     12592.31|     VND|Internet Banking|             NULL|Completed|REF931434515|     Fee transaction|2025-11-04|
|TXN0006758|ACCT009332|2025-11-04 20:34:28|        Fee|     61861.42|     VND|             POS|             NULL|Completed|REF461026093|     Fee transaction|2025-11-04|
|TXN0007974|ACCT014643|2025-11-04 01:26:45| Withdrawal|3.703897104E7|     VND|             ATM|             NULL|Completed|REF137355284|Withdrawal transa..

## 1. Running Balance per Account

In [5]:
from pyspark.sql.window import Window
# Define window: partition by account, order by date

account_window = Window.partitionBy("account_id").orderBy("txn_datetime").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate running total
txn_running = txn.withColumn("running_total", F.sum("amount").over(account_window))
txn_running.select("account_id", "txn_datetime", "txn_type", "amount", "running_total").show(15)

+----------+-------------------+------------+-------------+--------------------+
|account_id|       txn_datetime|    txn_type|       amount|       running_total|
+----------+-------------------+------------+-------------+--------------------+
|ACCT000004|2025-01-04 06:44:09|         Fee|     61157.77|            61157.77|
|ACCT000004|2025-01-23 01:05:02|         Fee|     46603.25|  107761.01999999999|
|ACCT000004|2025-02-08 05:53:23|     Payment|   6666852.89|   6774613.909999999|
|ACCT000004|2025-02-13 21:20:04|     Deposit|   6666852.89|1.3441466799999999E7|
|ACCT000004|2025-02-14 11:15:22|         Fee|     67729.26|1.3509196059999999E7|
|ACCT000004|2025-05-31 08:51:18|     Deposit|   6666852.89|       2.017604895E7|
|ACCT000004|2025-07-03 15:05:51| Transfer In|   6666852.89|       2.684290184E7|
|ACCT000004|2025-07-19 23:16:16| Transfer In|   6666852.89|       3.350975473E7|
|ACCT000004|2025-08-16 22:46:16|  Withdrawal|   6666852.89|       4.017660762E7|
|ACCT000004|2025-10-25 18:06

## 2. Customer Ranking by Transaction Volume

In [ ]:
# Aggregate first
customer_txn = txn.join(accounts_df.select("account_id", "customer_id"), "account_id")
customer_volume = customer_txn.groupBy("customer_id").agg(sum("amount").alias("total_volume"), count("*").alias("txn_count"))

# Rank customers
rank_window = Window.orderBy(col("total_volume").desc())
customer_ranked = customer_volume.withColumn("volume_rank", rank().over(rank_window)).withColumn("row_num", row_number().over(rank_window))
customer_ranked.show(10)

## 3. 7-Day Moving Average

In [ ]:
# Daily aggregation first
daily_txn = txn.groupBy("account_id", "txn_date").agg(sum("amount").alias("daily_amount"))

# 7-day moving average
moving_window = Window.partitionBy("account_id").orderBy("txn_date").rowsBetween(-6, 0)
daily_with_ma = daily_txn.withColumn("moving_avg_7d", avg("daily_amount").over(moving_window))
daily_with_ma.orderBy("account_id", "txn_date").show(15)

## 4. Lag and Lead - Transaction Velocity

In [ ]:
# Previous and next transaction
velocity_window = Window.partitionBy("account_id").orderBy("txn_datetime")
txn_velocity = txn.withColumn("prev_amount", lag("amount", 1).over(velocity_window)).withColumn("next_amount", lead("amount", 1).over(velocity_window))
txn_velocity.select("account_id", "txn_datetime", "amount", "prev_amount", "next_amount").show(10)

## 5. First and Last Transaction per Account

In [ ]:
fl_window = Window.partitionBy("account_id").orderBy("txn_datetime")
fl_window_desc = Window.partitionBy("account_id").orderBy(col("txn_datetime").desc())

first_last = txn.withColumn("first_txn", first("amount").over(fl_window)).withColumn("last_txn", first("amount").over(fl_window_desc))
first_last.select("account_id", "txn_datetime", "amount", "first_txn", "last_txn").show(10)

## Practice Exercises
1. Rank branches by total transaction amount (per region)
2. Calculate month-over-month growth rate per account
3. Find the top 3 transactions per customer by amount

In [ ]:
# Exercise: Top 3 transactions per customer
# Hint: Use row_number() with partition by customer, order by amount desc, then filter
# Your code here:


In [ ]:
spark.stop()